# Mesozoic Labs: Triceratops 60 cm, walking-policy training (simulation only)

This notebook trains a PPO walking policy for the 60 cm Triceratops build model in MuJoCo on a **CPU** runtime and keeps every file on Google Drive under `MyDrive/mesozoic-labs/triceratops60/<RUN_ID>`. **Simulation only: nothing here touches a robot.**

Follow the step sheet `hardware/triceratops_build/rl/COLAB_STEPS.md`. The first run is a **pipeline test** (about one hour, 2 million steps): it proves the install, the saves on Drive, the stop and the resume. It does not train a walking policy.

* Nothing runs when the notebook opens. Training starts only when cell 5 runs (Run all runs it), and only after cells 1 to 4 have checked the parameters, the code version, Python and the packages, and the run folder.
* Cell 5 runs `rl/train.py` in a subprocess and always resumes from the newest intact save in the run folder. To continue after a stop or a disconnect, run cell 5 again (after a disconnect: Run all) with the same values.
* The stop button on cell 5 makes `train.py` save once more and exit; Drive is then flushed. Click it once and wait for the line that begins `Session interrupted`: further clicks are ignored until the save is done.
* Runtime type: CPU with High-RAM. A GPU adds nothing to this training.
* Untested on Colab itself: the cells were run locally (outside Colab), where the Colab-only parts (fetching the commit, pip, Drive, disconnect) are skipped. The shallow fetch was tested against a local copy of the repository.

In [ ]:
# ===== 1. PARAMETERS: COLAB_STEPS.md gives every value; change nothing else =====
import json
import os
import re

REPO_REF = ""  # @param {"type":"string"}
RUN_ID = ""  # @param {"type":"string"}
MODEL = "trike60_C_closed"  # @param ["trike60_C_closed", "trike60_C_closed_ankleroll", "trike60_C_closed_capsule"]
TOTAL_STEPS = 2000000  # @param {"type":"integer"}
N_ENVS = 8  # @param {"type":"integer"}
SEED = 0  # @param {"type":"integer"}
ENV_PRESET = ""  # @param ["", "run3_terrain"]
EVAL_EPISODES = 5  # @param {"type":"integer"}
USE_GOOGLE_DRIVE = True  # @param {"type":"boolean"}
AUTO_DISCONNECT = True  # @param {"type":"boolean"}
# REPO_REF: the 40 hex characters of the commit to run.  RUN_ID: letters, digits, "-" and "_", such as
# trike60-pipe1; a new RUN_ID starts a new run, the same RUN_ID resumes it.  TOTAL_STEPS: the run's total
# environment steps; a run keeps it (to train longer, start a new RUN_ID).  N_ENVS: parallel simulations, one per
# vCPU (8 on Colab's High-RAM CPU runtime); a run keeps it.  MODEL: the robot model file (the foot: PR #669's,
# the roll ankle, or the capsule sole); a run keeps it.
# ENV_PRESET: "" for the environment's defaults (flat floor), or a settings file in rl/configs/ (run3_terrain: steps,
# ramps, pushes, low friction, a sideways-drift cost); a run keeps it.

NAMES = ("REPO_REF", "RUN_ID", "MODEL", "TOTAL_STEPS", "N_ENVS", "SEED", "ENV_PRESET", "EVAL_EPISODES",
         "USE_GOOGLE_DRIVE", "AUTO_DISCONNECT")
PARAMS = {name: globals()[name] for name in NAMES}
if os.environ.get("TRIKE60_PARAMS"):  # local runs and tests only: a JSON file whose values replace the ones above
    with open(os.environ["TRIKE60_PARAMS"], encoding="utf-8") as _f:
        PARAMS.update(json.load(_f))
    globals().update({name: PARAMS[name] for name in NAMES})

if not re.fullmatch(r"[0-9a-f]{40}", str(REPO_REF)):
    raise ValueError("REPO_REF must be the 40 hex characters of a commit (COLAB_STEPS.md gives it). Nothing was run.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", str(RUN_ID)):
    raise ValueError("RUN_ID must be letters, digits, '-' and '_' (for example trike60-pipe1). Nothing was run.")
if MODEL not in ("trike60_C_closed", "trike60_C_closed_ankleroll", "trike60_C_closed_capsule"):
    raise ValueError("MODEL must be trike60_C_closed, trike60_C_closed_ankleroll or trike60_C_closed_capsule. "
                     "Nothing was run.")
if ENV_PRESET not in ("", "run3_terrain"):
    raise ValueError("ENV_PRESET must be empty or run3_terrain. Nothing was run.")
if int(TOTAL_STEPS) <= 0 or int(N_ENVS) < 1 or int(EVAL_EPISODES) < 1:
    raise ValueError("TOTAL_STEPS, N_ENVS and EVAL_EPISODES must be positive. Nothing was run.")
print("Parameters of this session:")
for _name in NAMES:
    print(f"  {_name} = {PARAMS[_name]!r}")

In [ ]:
# ===== 2. CODE: the repository at exactly REPO_REF (that one commit, fetched without history) =====
import pathlib
import signal
import subprocess
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
REPO_URL = "https://github.com/kuds/mesozoic-labs.git"


def run_step(cmd, cwd=None, env=None):
    """Run cmd with its output shown here line by line.  It runs in its own session, so a notebook interrupt
    (which signals this kernel's whole process group) does not reach it directly.  On the stop button this function
    sends it ONE SIGINT (train.py then saves a checkpoint pair and exits), keeps showing its lines until it has
    exited, ignoring further clicks, and only then re-raises: nothing after it runs while the step is still saving."""
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            cwd=cwd, env=env, start_new_session=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        return proc.wait()
    except KeyboardInterrupt as stop:
        print("\nStop requested: the step is asked to stop (train.py first saves a checkpoint pair). Wait for its "
              "last line; further clicks are ignored.", flush=True)
        proc.send_signal(signal.SIGINT)
        while True:
            try:
                for line in proc.stdout:
                    print(line, end="", flush=True)
                proc.wait()
                break
            except KeyboardInterrupt:
                print("(the step is still saving: this click is ignored; please wait)", flush=True)
        raise stop


def git(*args, cwd=None):
    return subprocess.check_output(["git", *args], cwd=cwd or REPO_DIR, text=True).strip()


def checkout_commit(repo_dir, url, ref):
    """Exactly commit ``ref`` of ``url`` in ``repo_dir``: a shallow fetch of that commit only (no history), then a
    detached checkout.  Returns (HEAD, clean)."""
    repo_dir = pathlib.Path(repo_dir)
    if not (repo_dir / ".git").exists():
        repo_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet", str(repo_dir)], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "remote", "add", "origin", url], check=True)
    if (repo_dir / ".git" / "index").exists() and git("status", "--porcelain", "--untracked-files=no", cwd=repo_dir):
        raise RuntimeError("The checkout has local edits; use a fresh runtime (Runtime > Disconnect and delete runtime).")
    if run_step(["git", "-C", repo_dir, "fetch", "--quiet", "--depth", "1", "origin", ref]) != 0:
        raise RuntimeError(f"git could not fetch {ref}; check REPO_REF against COLAB_STEPS.md.")
    fetched = git("rev-parse", "FETCH_HEAD^{commit}", cwd=repo_dir)
    if run_step(["git", "-C", repo_dir, "checkout", "--quiet", "--detach", fetched]) != 0:
        raise RuntimeError("git checkout failed; read the lines above.")
    return git("rev-parse", "HEAD", cwd=repo_dir), not git("status", "--porcelain", "--untracked-files=no", cwd=repo_dir)


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/mesozoic-labs")
    HEAD, CLEAN = checkout_commit(REPO_DIR, REPO_URL, REPO_REF)
    print(f"Repository commit: {HEAD} (matches REPO_REF: {'yes' if HEAD == REPO_REF else 'NO'}; "
          f"checkout clean: {'yes' if CLEAN else 'NO'})")
    if HEAD != REPO_REF or not CLEAN:
        raise RuntimeError("The checkout is not exactly REPO_REF. Stop and report this line.")
else:
    REPO_DIR = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                    if (p / "hardware" / "triceratops_build" / "rl" / "train.py").exists())
    HEAD = git("rev-parse", "HEAD")
    print(f"Not on Colab: using the checkout at {REPO_DIR} (commit {HEAD}); REPO_REF only selects the Colab checkout.")
RL_DIR = REPO_DIR / "hardware" / "triceratops_build" / "rl"
# every step below is a subprocess with this environment: no OpenGL, one thread per process
STEP_ENV = dict(os.environ, MUJOCO_GL="disable", PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")

In [ ]:
# ===== 3. PYTHON AND PACKAGES: CPython 3.12-3.14; the pins of rl/requirements-colab.txt, torch as the CPU build =====
import platform

if platform.python_implementation() != "CPython" or not ((3, 12) <= sys.version_info[:2] <= (3, 14)):
    raise RuntimeError(f"This runtime has {platform.python_implementation()} {platform.python_version()}; the pinned "
                       "packages need CPython 3.12, 3.13 or 3.14. Nothing was installed. Stop and report this line.")
print(f"Python {platform.python_version()} (CPython 3.12-3.14 needed): yes")

REQ = RL_DIR / "requirements-colab.txt"
TORCH_CPU_INDEX = "https://download.pytorch.org/whl/cpu"
PINS = {}
for _line in REQ.read_text().splitlines():
    _line = _line.split("#", 1)[0].strip()
    if _line:
        _name, _version = _line.split("==")
        PINS[_name.strip()] = _version.strip()


def installed_versions():
    """The installed versions, read in a fresh interpreter (this kernel may still hold the old ones)."""
    code = ("import importlib.metadata as m, json, sys\nout = {}\nfor n in sys.argv[1:]:\n    try:\n"
            "        out[n] = m.version(n)\n    except m.PackageNotFoundError:\n        out[n] = None\nprint(json.dumps(out))")
    return json.loads(subprocess.check_output([sys.executable, "-c", code, *PINS], text=True))


def differences(vers):
    return [f"{n}: installed {vers.get(n)}, pinned {v}" for n, v in PINS.items()
            if (vers.get(n) or "").split("+", 1)[0] != v]


_before = installed_versions()
if IN_COLAB:
    if _before.get("torch") != f"{PINS['torch']}+cpu":  # pip would take an installed 2.x.y+cu... as ==2.x.y
        if run_step([sys.executable, "-m", "pip", "install", "-q", f"torch=={PINS['torch']}+cpu",
                     "--index-url", TORCH_CPU_INDEX]) != 0:
            raise RuntimeError("torch (CPU build) did not install; send the lines above.")
    if differences(installed_versions()):
        if run_step([sys.executable, "-m", "pip", "install", "-q", "-r", REQ]) != 0:
            raise RuntimeError("A pinned package did not install; send the lines above.")
VERSIONS = installed_versions()
_wrong = differences(VERSIONS)
if IN_COLAB and not (VERSIONS.get("torch") or "").endswith("+cpu"):
    _wrong.append(f"torch: installed {VERSIONS.get('torch')}, not the CPU build")
print("Packages: " + ", ".join(f"{n} {VERSIONS.get(n)}" for n in PINS))
if _wrong and IN_COLAB:
    raise RuntimeError("Pins do not hold: " + "; ".join(_wrong))
print("All pins hold: " + ("yes" if not _wrong else "NO (not on Colab, so not enforced): " + "; ".join(_wrong)))

In [ ]:
# ===== 4. STORAGE: Google Drive and the run folder =====
DRIVE_MOUNT = "/content/drive"


def ensure_drive():
    """On Colab with Drive: mount it unless it is mounted (cells 5 and 6 call this again, after a stop)."""
    if IN_COLAB and USE_GOOGLE_DRIVE and not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive

        drive.mount(DRIVE_MOUNT)
    if IN_COLAB and USE_GOOGLE_DRIVE and not os.path.ismount(DRIVE_MOUNT):
        raise RuntimeError("Google Drive is not mounted; nothing was started. Run this cell again and allow Drive.")


def flush_drive():
    """On Colab with Drive: push the pending writes to Drive (flush_and_unmount; ensure_drive mounts it again).
    The stop button is ignored while it runs: an interrupted unmount would leave Drive in an unknown state."""
    if IN_COLAB and USE_GOOGLE_DRIVE and os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive

        try:
            previous = signal.signal(signal.SIGINT, signal.SIG_IGN)
        except ValueError:  # not the main thread: cannot change the handler
            previous = None
        try:
            print("Flushing Google Drive writes...", flush=True)
            drive.flush_and_unmount()
            print("Drive flushed and unmounted; the next run of cell 5 mounts it again.", flush=True)
        finally:
            if previous is not None:
                signal.signal(signal.SIGINT, previous)


ensure_drive()
if IN_COLAB and USE_GOOGLE_DRIVE:
    BASE = pathlib.Path(DRIVE_MOUNT) / "MyDrive" / "mesozoic-labs" / "triceratops60"
elif IN_COLAB:
    BASE = pathlib.Path("/content/triceratops60")
    print(f"Warning: USE_GOOGLE_DRIVE is off. The files go to {BASE} and are lost with the runtime.")
else:
    BASE = pathlib.Path(PARAMS.get("LOCAL_RUNS_DIR") or pathlib.Path.home() / "mesozoic-labs-runs" / "triceratops60")
RUN_DIR = BASE / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
_record = RUN_DIR / "run_record.json"
if _record.exists():
    try:
        _sessions = json.loads(_record.read_text()).get("sessions") or []
    except ValueError as _exc:  # train.py moves an unreadable record aside and says so
        print(f"Note: {_record.name} could not be read ({_exc}); train.py will move it aside.")
        _sessions = []
    _first = (_sessions[0].get("git") or {}).get("commit") if _sessions else None
    if _first and _first != HEAD and IN_COLAB:
        raise RuntimeError(f"Run {RUN_ID} was started from commit {_first}, not {HEAD}. A run keeps its commit: "
                           "use the run's own REPO_REF, or a new RUN_ID. Nothing was changed.")
    if _first and _first != HEAD:
        print(f"Note: run {RUN_ID} was started from commit {_first}; this checkout is at {HEAD} (not enforced "
              "outside Colab).")
_ckpts = sorted((RUN_DIR / "checkpoints").glob("trike60_*_steps.zip"))
print(f"Run folder: {RUN_DIR} ({'resuming: ' + str(len(_ckpts)) + ' checkpoint(s) found' if _ckpts else 'new run'})")

In [ ]:
# ===== 5. TRAINING: rl/train.py in a subprocess; it resumes from the newest intact save in the run folder =====
ensure_drive()
# --exit-with-parent <this kernel's process id>: train.py stops (and saves) if this kernel dies or restarts
TRAIN_CMD = [sys.executable, "-u", RL_DIR / "train.py", "--out", RUN_DIR, "--total-steps", int(TOTAL_STEPS),
             "--n-envs", int(N_ENVS), "--seed", int(SEED), "--model", MODEL, "--resume", "--exit-with-parent",
             os.getpid()]
if ENV_PRESET:  # a settings file of this commit, rl/configs/<ENV_PRESET>.json
    _preset = RL_DIR / "configs" / f"{ENV_PRESET}.json"
    if not _preset.exists():
        raise RuntimeError(f"This commit has no {_preset.name} in rl/configs; check REPO_REF. Nothing was started.")
    TRAIN_CMD += ["--env-config", _preset]
print("Running: train.py " + " ".join(str(c) for c in TRAIN_CMD[3:]), flush=True)
try:
    _code = run_step(TRAIN_CMD, cwd=REPO_DIR, env=STEP_ENV)
except KeyboardInterrupt:
    flush_drive()  # the stop button: run_step returns only after train.py has saved and exited
    raise
if _code != 0:
    flush_drive()  # whatever train.py saved before it stopped
    raise RuntimeError(f"train.py stopped with exit code {_code}; the reason is in the lines above. "
                       "What it saved is in the run folder.")

In [ ]:
# ===== 6. EVALUATION: randomisation on, delay 10 ms (and 0 ms), plus a sweep of fixed commands at 10 ms =====
ensure_drive()
EVAL_CMD = [sys.executable, "-u", RL_DIR / "evaluate.py", "--run-dir", RUN_DIR, "--episodes", int(EVAL_EPISODES),
            "--delays", 10, 0, "--sweep"]
_code = run_step(EVAL_CMD, cwd=REPO_DIR, env=STEP_ENV)
if _code != 0:
    flush_drive()
    raise RuntimeError(f"evaluate.py stopped with exit code {_code}; the reason is in the lines above.")
print(f"Results: {RUN_DIR / 'eval'}")

In [ ]:
# ===== 7. FINISH: flush Drive and release the runtime =====
try:
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    from environments.shared.notebook_runtime import disconnect_runtime
except Exception as _exc:  # this kernel's own packages may be stale after cell 3: release the runtime directly
    print(f"(the shared helper did not import here: {_exc!r}; releasing the runtime directly)")

    def disconnect_runtime(reason, *, in_colab, auto, flush_drive):
        import time

        print(f"\n{reason}")
        if not (in_colab and auto):
            print("Auto-disconnect skipped (not in Colab or AUTO_DISCONNECT is False).")
            return
        if flush_drive:
            from google.colab import drive

            print("Flushing Google Drive writes...")
            drive.flush_and_unmount()
        from google.colab import runtime

        print("Disconnecting runtime in 5 seconds...")
        time.sleep(5)
        runtime.unassign()

disconnect_runtime(f"Run {RUN_ID} is finished; its files are in {RUN_DIR}.", in_colab=IN_COLAB, auto=AUTO_DISCONNECT,
                   flush_drive=IN_COLAB and USE_GOOGLE_DRIVE and os.path.ismount(DRIVE_MOUNT))